[Opijnen et al. 2009](https://doi.org/10.1038/nmeth.1377) tested fitness measurements for a series of ≈30 single gene knockouts in *Streptococcus pneumoniae*.\
We have digitized the data from Fig. 3 of the original paper and replot the comparision as a correlation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
import os
import os.path
from os import path

## create export directory if necessary
## foldernames for output plots/lists produced in this notebook
import os
FIG_DIR = f'./figures/opijnen_data/'
os.makedirs(FIG_DIR, exist_ok=True)
print("All  plots will be stored in: \n" + FIG_DIR)

In [ ]:
%run setup_aesthetics.py

## Load data

In [ ]:
# load data
df = pd.read_csv('./data/Opijnen2009/2009_Opijnen_Fig3.csv')
## show first entries
df.head(3)

In [ ]:
## format into two columns
df['1x1'] = df['Unnamed: 1']
df['Tn-seq'] = df['Unnamed: 3']
## drop superfluous columns and rows
df = df.drop(['Unnamed: 1', 'Unnamed: 3'], axis = 1, )
df = df.drop([0], axis = 0)

## convert to float
df = df.astype('float')

In [ ]:
## show converted entries
df.head(3)

In [ ]:
## subtract constant
df = df -1

## Plot correlation

In [ ]:
## fit a line through zero
from scipy.optimize import curve_fit

x = df['1x1'].values
y = df['Tn-seq'].values

lin = lambda x, a: a * x 
slope0 = curve_fit(lin, x, y)[0][0]
# slope is ≈1+average relative error assuming relative error is constant
print(slope0)

In [ ]:
from scipy.stats import linregress
x = df['1x1'].values
y = df['Tn-seq'].values
slope, intercept,_,_,_ = linregress(x,y)

In [ ]:
slope

In [ ]:
## plot as correlation
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['1x1'].values
y = df['Tn-seq'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
xymin, xymax = np.min([xmin,ymin]), np.max([xmax,ymax])
ax.set_xlim(xymin, xymax)
ax.set_ylim(xymin,xymax)


xvec = np.linspace(xymin,xymax, num = 30)
#ax.plot(xvec,xvec*slope+intercept)

ax.plot([xymin,xymax], [xymin,xymax], ls = '--', color = 'black')

ax.set_ylabel('mutant fitness in bulk competition')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Opijnen et al. 2009 (n = {df.shape[0]} knockouts)', loc = 'right')

#fig.tight_layout()
fig.savefig(FIG_DIR+ 'fitness_pairwise_vs_fitness_bulk.pdf',
            DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Plot absolute error

In [ ]:
### plot absolute error
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['1x1'].values
y = df['Tn-seq'].values - df['1x1'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 

ymin, ymax = ax.get_ylim()
yabsmax = np.max(np.abs([ymin,ymax]))
ax.set_ylim(-yabsmax,yabsmax)
ax.set_xlim(xymin,xymax)


ax.axhline(0, ls = '--', color = 'black')

ax.set_ylabel('absolute error')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Opijnen et al. 2009 (n = {df.shape[0]} knockouts)', loc = 'right')


#fig.tight_layout()
fig.savefig(FIG_DIR + 'absolute_error_bulk_competition.pdf',\
             DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
## calculate correlation
from scipy.stats import pearsonr

r, p = pearsonr(x,y)
print(r)
print(p)

### Plot relative error

In [ ]:
### Plot histogram of pairwise fitness values

x = np.abs( df['1x1'].values)
fig,ax = plt.subplots()
_ = ax.hist(x, bins = 20)

In [ ]:
## compute relative error

x = df['1x1'].values
delta = df['Tn-seq'].values - df['1x1'].values
y = np.abs(np.divide(delta, x, where = x!= 0))

## remove entries with zero relative erro
is_zero = y == 0
y = y[~is_zero]
x = x[~is_zero]

In [ ]:
## calculate mean relative error 
is_not_neutral = np.abs(x) > 0.1
rel_error_mean = np.exp(np.log(y[is_not_neutral]).mean())

In [ ]:
### plot relative error
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))


ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 
ax.scatter(x[is_not_neutral],y[is_not_neutral], rasterized = True, color = 'blue')

ax.set_xlim(xymin,xymax)
ax.set_yscale('log')
#ax.set_ylim(ymin=1e-3)

#ax.axhline(0, ls = '--', color = 'black')
ax.axhline(rel_error_mean)

ax.set_ylabel('relative error')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Opijnen et al. 2009 (n = {df.shape[0]} knockouts)', loc = 'right')


#fig.tight_layout()
fig.savefig(FIG_DIR + 'relative_error_bulk_competition_with_rel_error_trend.pdf',\
             DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
print(rel_error_mean)

In [ ]:
y[is_not_neutral]

### Plot correlation with relative error trend

In [ ]:
is_not_neutral = np.abs(df['1x1']) >0.1

In [ ]:

## plot as correlation
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['1x1'].values
y = df['Tn-seq'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 
ax.scatter(x[is_not_neutral],y[is_not_neutral], rasterized = True, color = 'blue')

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
xymin, xymax = np.min([xmin,ymin]), np.max([xmax,ymax])
ax.set_xlim(xymin, xymax)
ax.set_ylim(xymin,xymax)

xvec = np.linspace(xymin,xymax, num = 30)
ax.plot(xvec,xvec*(1+rel_error_mean), color = 'tab:blue')
ax.plot(xvec,xvec*(1-rel_error_mean), color = 'tab:blue')

ax.plot([xymin,xymax], [xymin,xymax], ls = '--', color = 'black')

ax.set_ylabel('mutant fitness in bulk competition')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Opijnen et al. 2009 (n = {df.shape[0]} knockouts)', loc = 'right')

#fig.tight_layout()
fig.savefig(FIG_DIR+ 'fitness_pairwise_vs_fitness_bulk_with_rel_error_trend.pdf',
            DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Plot absolute error with relative error trend

In [ ]:

### plot absolute error
fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

x = df['1x1'].values
y = df['Tn-seq'].values - df['1x1'].values

ax.scatter(x,y, rasterized = True, color = 'silver', marker = 'o') 
ax.scatter(x[is_not_neutral],y[is_not_neutral], rasterized = True, color = 'blue')

ymin, ymax = ax.get_ylim()
yabsmax = np.max(np.abs([ymin,ymax]))
ax.set_ylim(-yabsmax,yabsmax)
ax.set_xlim(xymin,xymax)

xvec = np.linspace(xymin,xymax, num = 30)
#ax.plot(xvec,xvec*(rel_error_mean))

ax.axhline(0, ls = '--', color = 'black')

ax.set_ylabel('absolute error')
ax.set_xlabel('mutant fitness in pairwise competition')
ax.set_title(f'Opijnen et al. 2009 (n = {df.shape[0]} knockouts)', loc = 'right')


#fig.tight_layout()
fig.savefig(FIG_DIR + 'absolute_error_bulk_competition_with_rel_error_trend.pdf',\
             DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
## calculate correlation
from scipy.stats import pearsonr

r, p = pearsonr(x[is_not_neutral],y[is_not_neutral])
print(r)
print(p)